<a href="https://colab.research.google.com/github/keyvanmahmoudi/Deep_Learning_Course/blob/main/solutions/Session09_transfer_learning_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۹: یادگیری انتقالی (Transfer Learning) و Fine-tuning

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session09_transfer_learning.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

> **نکتهٔ اجرا:** بعضی پاسخ‌ها مدل‌های بزرگ‌تری آموزش می‌دهند؛ در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
import copy
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
print('device:', device)

IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.6, 1.0)), transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                              transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

train_set = datasets.Flowers102('data', split='train', download=True, transform=train_tf)
val_set = datasets.Flowers102('data', split='val', download=True, transform=eval_tf)
test_set = datasets.Flowers102('data', split='test', download=True, transform=eval_tf)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=128, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=128, num_workers=2, pin_memory=True)
N_CLASSES, EPOCHS = 102, 15


def freeze_bn(model):
    """Keep every BatchNorm layer in eval mode (running statistics are not updated)."""
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()


def train_one_epoch(model, loader, criterion, optimizer, scaler, scheduler=None, bn_eval=False):
    model.train()
    if bn_eval:
        freeze_bn(model)
    correct = n = 0
    for X, y in loader:
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(X)
            loss = criterion(logits, y)
        optimizer.zero_grad()
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        if scheduler is not None:
            scheduler.step()
        correct += (logits.argmax(1) == y).sum().item(); n += len(y)
    return correct / n


@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct = n = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            correct += (model(X).argmax(1) == y).sum().item()
        n += len(y)
    return correct / n


def run_experiment(model, param_groups, epochs, label, label_smoothing=0.1, bn_eval=False):
    """AdamW + cosine schedule; returns the model with the best validation weights."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(train_loader))
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    best_acc, best_state, start = 0.0, None, time.time()
    for _ in range(epochs):
        train_one_epoch(model, train_loader, criterion, optimizer, scaler, scheduler, bn_eval)
        va = accuracy(model, val_loader)
        if va > best_acc:
            best_acc, best_state = va, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    elapsed = time.time() - start
    print(f'{label}: best val acc {best_acc:.4f} | test acc {accuracy(model, test_loader):.4f} | {elapsed:.0f}s')
    return model, elapsed


def new_head(model, attr_path):
    """Replace the last Linear layer (e.g. 'fc' or 'classifier.1') with a 102-class head."""
    parent, name = model, attr_path
    if '.' in attr_path:
        first, name = attr_path.split('.')
        parent = getattr(model, first)
        old = parent[int(name)]
        parent[int(name)] = nn.Linear(old.in_features, N_CLASSES)
        return parent[int(name)]
    old = getattr(model, name)
    setattr(model, name, nn.Linear(old.in_features, N_CLASSES))
    return getattr(model, name)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

به‌جای ResNet-18 از `efficientnet_b0` و `resnet50` استفاده کنید (نام لایه آخر EfficientNet `classifier` است). دقت، زمان آموزش و تعداد پارامتر را مقایسه کنید.

</div>

In [ ]:
candidates = {
    'resnet18': (models.resnet18, models.ResNet18_Weights.IMAGENET1K_V1, 'fc'),
    'resnet50': (models.resnet50, models.ResNet50_Weights.IMAGENET1K_V2, 'fc'),
    'efficientnet_b0': (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1, 'classifier.1'),
}
for name, (builder, w, head_path) in candidates.items():
    torch.manual_seed(0)
    model = builder(weights=w)
    head = new_head(model, head_path)
    head_ids = {id(p) for p in head.parameters()}
    groups = [{'params': [p for p in model.parameters() if id(p) not in head_ids], 'lr': 1e-4},
              {'params': head.parameters(), 'lr': 1e-3}]
    print(f'{name}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters')
    run_experiment(model, groups, EPOCHS, name)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

در `efficientnet_b0` طبقه‌بند یک `nn.Sequential(Dropout, Linear)` است؛ پس لایهٔ آخر `classifier[1]` است. انتظار داریم:

- **EfficientNet-B0** با حدود ۵ میلیون پارامتر (کمتر از ResNet-18) دقتی برابر یا بهتر بدهد؛ این شبکه با جست‌وجوی معماری (NAS) و مقیاس‌دهی متوازن عمق، عرض و وضوح طراحی شده است.
- **ResNet-50** (حدود ۲۵ میلیون پارامتر) معمولاً دقیق‌تر از ResNet-18 است، به قیمت زمان آموزش و حافظهٔ بیشتر.

مقایسهٔ زمان‌ها هم آموزنده است: تعداد پارامتر تنها معیار هزینه نیست. EfficientNet با پارامتر کم، به‌خاطر کانولوشن‌های Depthwise، روی GPU لزوماً سریع‌تر از ResNet-18 نیست.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

**Gradual Unfreezing** را پیاده کنید: ۵ Epoch فقط `fc`، سپس `layer4` را هم آزاد کنید، و در ۵ Epoch آخر همه لایه‌ها را.

</div>

In [ ]:
w = models.ResNet18_Weights.IMAGENET1K_V1
torch.manual_seed(0)
model = models.resnet18(weights=w)
model.fc = nn.Linear(model.fc.in_features, N_CLASSES)
model = model.to(device)


def set_trainable(model, prefixes):
    for name, p in model.named_parameters():
        p.requires_grad = any(name.startswith(pr) for pr in prefixes)


stages = [(['fc.'], 5, 1e-3), (['fc.', 'layer4.'], 5, 3e-4), ([''], 5, 1e-4)]    # '' matches every name
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler(enabled=USE_AMP)
for prefixes, epochs, lr in stages:
    set_trainable(model, prefixes)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)          # a fresh optimizer for each stage
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(train_loader))
    for _ in range(epochs):
        train_one_epoch(model, train_loader, criterion, optimizer, scaler, scheduler)
    print(f'trainable {sum(p.numel() for p in params):>10,} params ({prefixes}) -> val acc {accuracy(model, val_loader):.4f}')
print(f'gradual unfreezing test acc: {accuracy(model, test_loader):.4f}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

در هر مرحله یک Optimizer تازه فقط با پارامترهای قابل آموزش همان مرحله می‌سازیم. نرخ یادگیری را هم با باز شدن لایه‌های بیشتر کم می‌کنیم. ایده: وقتی Head هنوز تصادفی است، گرادیان‌های بزرگ و بی‌معنای آن ممکن است ویژگی‌های ارزشمند لایه‌های پیش‌آموزش‌دیده را خراب کنند (**Catastrophic Forgetting**). پس اول Head به یک نقطهٔ معقول می‌رسد، بعد لایه‌های عمیق‌تر به‌تدریج باز می‌شوند. این روش (Howard & Ruder, 2018، ULMFiT) معمولاً به نتیجه‌ای هم‌سطح یا کمی بهتر از Fine-tuning یکجا با Discriminative LR می‌رسد. مزیت اصلی آن پایداری بیشتر روی داده‌های بسیار کم است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

در حالت Feature Extraction، یک بار لایه‌های BatchNorm را در حالت `eval` نگه دارید (بعد از `model.train()` روی ماژول‌های `nn.BatchNorm2d` متد `eval()` را صدا بزنید) و نتیجه را مقایسه کنید.

</div>

In [ ]:
w = models.ResNet18_Weights.IMAGENET1K_V1
for bn_eval in (False, True):
    torch.manual_seed(0)
    model = models.resnet18(weights=w)
    for p in model.parameters():
        p.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, N_CLASSES)
    label = 'feature extraction, BN ' + ('frozen (eval)' if bn_eval else 'updating (train)')
    run_experiment(model, [{'params': model.fc.parameters(), 'lr': 1e-3}], EPOCHS, label, bn_eval=bn_eval)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

وقتی Backbone منجمد است ولی مدل در حالت `train` است، BatchNorm هنوز **آمار در حال اجرای** خود را با دادهٔ Flowers به‌روز می‌کند. وزن‌ها ثابت‌اند اما ورودی لایه‌ها عوض می‌شود. Head یاد می‌گیرد با ویژگی‌هایی کار کند که با آمار Batchهای ۳۲تایی نرمال شده‌اند، ولی در ارزیابی با Running Stats جدید (و متفاوت) روبه‌رو می‌شود. با نگه داشتن BatchNorm در حالت `eval`، Backbone یک **استخراج‌کنندهٔ ویژگی کاملاً ثابت** است و ویژگی‌ها در آموزش و ارزیابی یکسان‌اند. معمولاً این حالت کمی بهتر یا دست‌کم پایدارتر است، به‌ویژه با Batchهای کوچک.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

با ۴۰ تصویر دلخواه از دو یا سه دسته (مثلاً عکس‌های گوشی خودتان) یک Dataset با `datasets.ImageFolder` بسازید و یک طبقه‌بند با Fine-tuning آموزش دهید.

</div>

In [ ]:
# Folder layout expected by ImageFolder (one sub-folder per class):
#   my_photos/
#       cat/      img001.jpg ...
#       dog/      ...
#       flower/   ...
import os
from torch.utils.data import Subset

DATA_DIR = 'my_photos'
if not os.path.isdir(DATA_DIR):
    print(f'Create the folder {DATA_DIR!r} with one sub-folder per class (about 40 photos in total), then run again.')
else:
    full_train = datasets.ImageFolder(DATA_DIR, transform=train_tf)
    full_eval = datasets.ImageFolder(DATA_DIR, transform=eval_tf)            # same files, no augmentation
    rng = np.random.default_rng(0)
    idx = rng.permutation(len(full_train))
    n_val = max(len(idx) // 4, len(full_train.classes))
    my_train, my_val = Subset(full_train, idx[n_val:]), Subset(full_eval, idx[:n_val])
    print('classes:', full_train.classes, '| train:', len(my_train), '| validation:', len(my_val))

    my_train_loader = DataLoader(my_train, batch_size=8, shuffle=True)
    my_val_loader = DataLoader(my_val, batch_size=16)
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, len(full_train.classes))
    model = model.to(device)
    backbone = [p for n, p in model.named_parameters() if not n.startswith('fc.')]
    optimizer = torch.optim.AdamW([{'params': backbone, 'lr': 1e-4}, {'params': model.fc.parameters(), 'lr': 1e-3}])
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    for epoch in range(10):
        train_one_epoch(model, my_train_loader, criterion, optimizer, scaler)
        print(f'epoch {epoch + 1}: validation accuracy {accuracy(model, my_val_loader):.3f}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

`ImageFolder` نام هر زیرپوشه را یک کلاس در نظر می‌گیرد (به ترتیب حروف الفبا). دو نکتهٔ مهم:

- دو نمونه از Dataset ساختیم، یکی با Augmentation برای آموزش و یکی بدون آن برای اعتبارسنجی، و با **یک جایگشت مشترک** تقسیم کردیم تا هیچ تصویری در هر دو بخش نباشد.
- با این تعداد کم تصویر، دقت اعتبارسنجی روی ۱۰ تصویر بسیار پرنوسان است (هر تصویر ۱۰٪!). برای ارزیابی مطمئن‌تر از Cross-Validation استفاده کنید. با تصاویری که تفاوت واضحی دارند، Fine-tuning معمولاً بعد از چند Epoch تقریباً همه را درست تشخیص می‌دهد؛ این قدرت ویژگی‌های ImageNet است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۵

دقت را بدون Label Smoothing (`label_smoothing=0`) تکرار کنید.

</div>

In [ ]:
w = models.ResNet18_Weights.IMAGENET1K_V1
for smoothing in (0.1, 0.0):
    torch.manual_seed(0)
    model = models.resnet18(weights=w)
    model.fc = nn.Linear(model.fc.in_features, N_CLASSES)
    backbone = [p for n, p in model.named_parameters() if not n.startswith('fc.')]
    groups = [{'params': backbone, 'lr': 1e-4}, {'params': model.fc.parameters(), 'lr': 1e-3}]
    run_experiment(model, groups, EPOCHS, f'fine-tuning, label_smoothing={smoothing}', label_smoothing=smoothing)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

Label Smoothing با $\varepsilon = 0.1$ هدف کلاس درست را از ۱ به $0.9 + 0.1/102$ کم می‌کند و بقیه را کمی بالا می‌برد. پس مدل هیچ‌وقت برای رسیدن به احتمال دقیقاً ۱ تشویق نمی‌شود و Logitها بی‌نهایت بزرگ نمی‌شوند. با فقط ۱۰ تصویر برای هر کلاس، این منظم‌سازی معمولاً یکی دو درصد دقت آزمون را بهتر می‌کند. اثر مهم‌تر آن روی **کالیبراسیون** است: احتمال‌های پیش‌بینی‌شده به دقت واقعی نزدیک‌تر می‌شوند. تفاوت دقیق به Seed وابسته است؛ برای نتیجه‌گیری مطمئن، آزمایش را با چند Seed تکرار کنید.

</div>